In [1]:
import pandas as pd
import numpy as np

## 1. Load dataset final

In [2]:
df = pd.read_parquet("dataset_final.parquet")
print(df.shape)
df.head(3)

(274790, 27)


,site_name,delivery_time,production,installed_capacity,wind_speed_10m,wind_speed_100m,wind_direction_10m,wind_direction_100m,wind_gusts_10m,temperature_2m,...,cloud_cover_low,cloud_cover_mid,cloud_cover_high,shortwave_radiation,direct_radiation,diffuse_radiation,weather_code,sunshine_duration,latitude,longitude
0,Belwind Phase 1,2023-01-01 00:00:00+00:00,147.7025,171.0,14.603082,19.897738,218.04709,219.28940,20.7,12.25,...,53.0,100.0,98.0,0.0,0.0,0.0,51.0,0.0,51.66,2.8
1,Belwind Phase 1,2023-01-01 01:00:00+00:00,146.1775,171.0,16.182089,21.681328,215.94937,217.50421,20.8,12.10,...,18.0,100.0,100.0,0.0,0.0,0.0,51.0,0.0,51.66,2.8
2,Belwind Phase 1,2023-01-01 02:00:00+00:00,146.1800,171.0,17.969420,23.809662,226.80397,228.74626,24.1,11.85,...,31.0,100.0,100.0,0.0,0.0,0.0,51.0,0.0,51.66,2.8


## 2. Target : capacity factor

Normalise la production par la capacité installée → valeur dans [0, 1].  
Permet au modèle global d'apprendre une seule courbe de puissance pour tous les parcs.

In [3]:
df["capacity_factor"] = df["production"] / df["installed_capacity"]
df["capacity_factor"] = df["capacity_factor"].clip(0, 1)

print("Target — capacity_factor")
print(df["capacity_factor"].describe().round(3))

Target — capacity_factor
count    274790.000
mean          0.364
std           0.344
min           0.000
25%           0.050
50%           0.239
75%           0.689
max           0.991
Name: capacity_factor, dtype: float64


## 3. Features temporelles

Encodage sin/cos pour préserver la cyclicité (ex : heure 23 ≈ heure 0).

**Décisions issues de l'EDA :**
- `month_sin/cos` : saisonnalité mensuelle forte (+40% hiver vs été) → **inclus**
- `hour_sin/cos` : profil horaire quasi-plat en offshore → inclus comme contexte
- `dayofweek` : aucun effet visible dans l'EDA → **exclu**
- `dayofyear` (sin/cos) : redondant avec `month` sur 3 ans de données → **exclu**

In [4]:
dt = df["delivery_time"]

df["hour"]  = dt.dt.hour
df["month"] = dt.dt.month

# Cyclic encoding
df["hour_sin"]  = np.sin(2 * np.pi * df["hour"]  / 24)
df["hour_cos"]  = np.cos(2 * np.pi * df["hour"]  / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

print("Features temporelles :", ["hour_sin","hour_cos","month_sin","month_cos"])
print("Exclus (EDA) : dayofweek (pas d'effet), dayofyear (redondant avec month)")

Features temporelles : ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']
Exclus (EDA) : dayofweek (pas d'effet), dayofyear (redondant avec month)


## 4. Features météo dérivées

### 4a. Composantes vectorielles du vent
La direction en degrés est circulaire (359° ≈ 1°) → décomposition en u/v.
Confirmé par l'EDA : `wind_direction` non corrélé directement, les composantes sont plus informatives.

### 4b. Proxy puissance (vent³)
La loi de Betz prédit P ∝ v³ dans la zone de ramp-up.
**Note EDA :** r(vent³, CF) = 0.757 < r(vent, CF) = 0.882 à cause de la saturation au plateau.
On l'inclut quand même — LightGBM est non-linéaire et en tire du signal dans la zone 4–12 m/s.

### 4c. Variables exclues après EDA
- `surface_pressure` : quasi-identique à `pressure_msl` offshore (corrélation ~1.0) → **exclu**
- `snowfall` : r = 0.028, quasi-nulle en mer du Nord → **exclu**

In [5]:
# Composantes vectorielles du vent à 10m et 100m
for height in [10, 100]:
    speed = df[f"wind_speed_{height}m"]
    direction_rad = np.deg2rad(df[f"wind_direction_{height}m"])
    df[f"wind_u_{height}m"] = -speed * np.sin(direction_rad)
    df[f"wind_v_{height}m"] = -speed * np.cos(direction_rad)

# Proxy puissance : vent³ à 100m (hauteur de nacelle offshore)
df["wind_power_proxy"] = df["wind_speed_100m"] ** 3

# Différence température / point de rosée (proxy humidité / risque de givrage)
df["temp_dew_diff"] = df["temperature_2m"] - df["dewpoint_2m"]

# surface_pressure et snowfall exclus (voir EDA)

print("Features météo dérivées :",
      ["wind_u_10m","wind_v_10m","wind_u_100m","wind_v_100m",
       "wind_power_proxy","temp_dew_diff"])
print("Exclus : surface_pressure (redondant), snowfall (r=0.028)")

Features météo dérivées : ['wind_u_10m', 'wind_v_10m', 'wind_u_100m', 'wind_v_100m', 'wind_power_proxy', 'temp_dew_diff']
Exclus : surface_pressure (redondant), snowfall (r=0.028)


## 5. Features identité du parc

Le modèle est global (1 modèle pour les 10 parcs).  
On encode l'identité du parc via ses caractéristiques physiques — plus expressif qu'un simple one-hot.

In [6]:
# Encodage catégoriel du site (requis par LightGBM)
df["site_id"] = df["site_name"].astype("category")

# La capacité installée et les coordonnées GPS sont déjà dans le dataset
# → latitude, longitude, installed_capacity directement utilisables comme features

print("Features site : site_id (catégorie), installed_capacity, latitude, longitude")
print()
print(df[["site_name","installed_capacity","latitude","longitude"]].drop_duplicates().to_string(index=False))

Features site : site_id (catégorie), installed_capacity, latitude, longitude

                       site_name  installed_capacity  latitude  longitude
                 Belwind Phase 1               171.0   51.6600     2.8000
             Mermaid Offshore WP               235.5   51.6300     2.8597
     Nobelwind Offshore Windpark               165.0   51.6631     2.8339
             Norther Offshore WP               369.6   51.5280     3.0320
                   Northwester 2               219.1   51.6875     2.7488
                       Northwind               216.0   51.6186     2.8998
              Rentel Offshore WP               307.0   51.5910     2.9440
             Seastar Offshore WP               252.0   51.7190     2.7400
Thorntonbank - C-Power - Area NE               147.6   51.5560     2.9689
Thorntonbank - C-Power - Area SW               177.6   51.5400     2.9210


## 6. Features de lag (historique de production)

On trie d'abord par (site_name, delivery_time) pour garantir l'ordre chronologique.
Les lags sont calculés **par parc** pour éviter de mélanger les séries.

**Décisions issues de l'EDA (ACF/PACF) :**

| Lag | Signal ACF | Décision |
|-----|-----------|----------|
| 24h | Pic net | → **inclus** (même heure la veille) |
| 48h | Pic net | → **inclus** (robustesse) |
| 168h | Pic modeste | → **inclus** (saisonnalité hebdo légère) |

> Ces lags sont légitimes en Day-Ahead : à l'heure de prédiction T, la production jusqu'à T est connue.

In [7]:
df = df.sort_values(["site_name", "delivery_time"]).reset_index(drop=True)

for lag_h in [24, 48, 168]:
    df[f"prod_lag_{lag_h}h"] = (
        df.groupby("site_name")["capacity_factor"]
          .shift(lag_h)
    )

print("Lags ajoutés : prod_lag_24h, prod_lag_48h, prod_lag_168h")
print(f"NaN introduits (normaux, début de série) : {df['prod_lag_24h'].isna().sum()}")

Lags ajoutés : prod_lag_24h, prod_lag_48h, prod_lag_168h
NaN introduits (normaux, début de série) : 240


## 7. Rolling features

Moyennes et écart-types glissants sur le vent 100m.
Capturent la tendance récente et la variabilité — complémentaires aux lags de production.

Fenêtres choisies : 3h (court terme), 6h (moyen terme), 24h (tendance journalière).

In [8]:
for window in [3, 6, 24]:
    df[f"wind_roll_mean_{window}h"] = (
        df.groupby("site_name")["wind_speed_100m"]
          .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f"wind_roll_std_{window}h"] = (
        df.groupby("site_name")["wind_speed_100m"]
          .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )

print("Rolling features ajoutées (fenêtres 3h, 6h, 24h) : mean + std du vent 100m")

Rolling features ajoutées (fenêtres 3h, 6h, 24h) : mean + std du vent 100m


## 8. Définition des listes de features

Listes finales basées sur les décisions de l'EDA.

In [9]:
METEO_RAW = [
    # Vent — features dominantes (r ~ 0.88)
    "wind_speed_10m", "wind_speed_100m",
    "wind_u_10m", "wind_v_10m", "wind_u_100m", "wind_v_100m",
    "wind_gusts_10m",
    "wind_power_proxy",          # vent³ — proxy loi de Betz
    # Température / humidité
    "temperature_2m", "dewpoint_2m", "temp_dew_diff", "apparent_temperature",
    # Pression (surface_pressure exclu — redondant avec pressure_msl offshore)
    "pressure_msl",
    # Précipitations (snowfall exclu — r=0.028)
    "precipitation",
    # Nuages
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    # Rayonnement
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "sunshine_duration",
]

TIME_FEATURES = [
    "hour_sin", "hour_cos",    # profil horaire (signal faible mais contexte utile)
    "month_sin", "month_cos",  # saisonnalité mensuelle forte
    # dayofweek exclu (aucun effet dans EDA)
    # dayofyear exclu (redondant avec month)
]

SITE_FEATURES = [
    "installed_capacity",  # différences de rendement inter-parcs (médiane 0.18–0.31)
    "latitude",
    "longitude",
    "site_id",             # encodage catégoriel LightGBM
]

LAG_FEATURES = [
    "prod_lag_24h",         # pic ACF net
    "prod_lag_48h",         # pic ACF net
    "prod_lag_168h",        # pic ACF modeste (hebdo)
    "wind_roll_mean_3h",
    "wind_roll_mean_6h",
    "wind_roll_mean_24h",
    "wind_roll_std_3h",
    "wind_roll_std_6h",
    "wind_roll_std_24h",
]

FEATURE_COLS = METEO_RAW + TIME_FEATURES + SITE_FEATURES + LAG_FEATURES
TARGET_COL   = "capacity_factor"

print(f"Total features : {len(FEATURE_COLS)}")
print(f"  Météo brutes + dérivées : {len(METEO_RAW)}")
print(f"  Temporelles             : {len(TIME_FEATURES)}")
print(f"  Site                    : {len(SITE_FEATURES)}")
print(f"  Lags / rolling          : {len(LAG_FEATURES)}")
print(f"\nExclus (EDA) : surface_pressure, snowfall, dayofweek, dayofyear")

Total features : 39
  Météo brutes + dérivées : 22
  Temporelles             : 4
  Site                    : 4
  Lags / rolling          : 9

Exclus (EDA) : surface_pressure, snowfall, dayofweek, dayofyear


## 10. Sauvegarde

In [10]:
import json
config = {
    "target": TARGET_COL,
    "features": FEATURE_COLS,
    "meteo": METEO_RAW,
    "time": TIME_FEATURES,
    "site": SITE_FEATURES,
    "lags": LAG_FEATURES,
}
with open("feature_config.json", "w") as f:
    json.dump(config, f, indent=2)

In [11]:
df.to_parquet("dataset_final_fe.parquet", index=False)